In [ ]:
from google.colab import drive
import shutil
import os

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Copy the zip to Colab's high-speed storage
print("Copying zip to local runtime...")
shutil.copy('/content/drive/MyDrive/dataset_a.zip', '/content/dataset_a.zip')

# 3. Unzip the file silently (-q) to prevent browser crashing from 32k print statements
print("Extracting dataset...")
!unzip -q /content/dataset_a.zip -d /content/

print("Extraction complete. Ready for processing.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying zip to local runtime...
Extracting dataset...
replace /content/dataset_a/ses_20260630-121953-LAPTOP-R36BQBTE/chunk_1200/screenshots/scr_smart_1782821994524_monitor_1_post.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Extraction complete. Ready for processing.


In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path

# Setup Device and Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights).to(device)
model.eval()
preprocess = weights.transforms()

# Custom Dataset to handle the nested directory
class ScreenshotDataset(Dataset):
    def __init__(self, root_dir):
        # Recursively find all PNGs/JPGs inside 'screenshots' folders
        self.image_paths = sorted(list(Path(root_dir).rglob('screenshots/scr_smart_*.*')))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        return preprocess(img), str(path)

# Initialize DataLoader
dataset_root = '/content/dataset_a'
dataset = ScreenshotDataset(dataset_root)
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Found {len(dataset)} screenshots. Commencing batch processing...")

Using compute device: cuda
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 113MB/s]


Found 34563 screenshots. Commencing batch processing...


In [ ]:
import torch.nn.functional as F
import json

previous_vector = None
segment_boundaries = []
threshold = 0.85

with torch.no_grad():
    for batch_imgs, batch_paths in dataloader:
        # Move batch to GPU and vectorize
        batch_imgs = batch_imgs.to(device)
        vectors = model(batch_imgs)

        # Iterate through the batch to calculate similarities
        for i in range(len(vectors)):
            current_vector = vectors[i]
            path = batch_paths[i]

            if previous_vector is not None:
                # Calculate Cosine Similarity (using PyTorch built-in function)
                similarity = F.cosine_similarity(previous_vector.unsqueeze(0), current_vector.unsqueeze(0)).item()

                if similarity < threshold:
                    segment_boundaries.append({
                        "boundary_detected_at": path,
                        "similarity_score": round(similarity, 4)
                    })

            previous_vector = current_vector

# Save results locally in Colab
output_file = '/content/vision_boundaries.jsonl'
with open(output_file, 'w') as f:
    for boundary in segment_boundaries:
        f.write(json.dumps(boundary) + '\n')

# Copy the final results back to your Google Drive so you don't lose them
shutil.copy(output_file, '/content/drive/MyDrive/vision_boundaries.jsonl')
print(f"Processing complete! Found {len(segment_boundaries)} state changes. File saved to Google Drive.")

Processing complete! Found 9481 state changes. File saved to Google Drive.


In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from collections import deque
import json
import shutil

# 1. Setup Device and Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights).to(device)
model.eval()
preprocess = weights.transforms()

# 2. Custom Dataset to handle the nested directory structure
class ScreenshotDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = sorted(list(Path(root_dir).rglob('screenshots/scr_smart_*.*')))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        img = Image.open(path).convert("RGB")
        return preprocess(img), str(path)

# Initialize DataLoader
dataset_root = '/content/dataset_a'
dataset = ScreenshotDataset(dataset_root)
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Found {len(dataset)} screenshots. Commencing advanced multi-frame processing...")

# 3. Rolling Window Buffer and Anomaly Detection Logic
window = deque(maxlen=4)
segment_boundaries = []
drop_tolerance = 0.15  # Minimum difference required between surrounding stability and transition

with torch.no_grad():
    for batch_imgs, batch_paths in dataloader:
        batch_imgs = batch_imgs.to(device)
        vectors = model(batch_imgs)

        for i in range(len(vectors)):
            current_vector = vectors[i]
            path = batch_paths[i]

            # Push current frame vector and path into the rolling window
            window.append((current_vector, path))

            # Once window is full (4 frames), compute multi-frame relational comparison
            if len(window) == 4:
                v1, p1 = window[0]
                v2, p2 = window[1]
                v3, p3 = window[2]
                v4, p4 = window[3]

                # Compute similarities across the window
                sim_before = F.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
                sim_transition = F.cosine_similarity(v2.unsqueeze(0), v3.unsqueeze(0)).item()
                sim_after = F.cosine_similarity(v3.unsqueeze(0), v4.unsqueeze(0)).item()

                # Dynamic baseline: Average stability of surrounding frames
                avg_stability = (sim_before + sim_after) / 2
                drop_magnitude = avg_stability - sim_transition

                # Anomaly condition: Transition dips significantly below the surrounding stability context
                if drop_magnitude > drop_tolerance:
                    segment_boundaries.append({
                        "boundary_detected_at": p3,
                        "drop_magnitude": round(drop_magnitude, 4),
                        "transition_similarity": round(sim_transition, 4),
                        "stability_before": round(sim_before, 4),
                        "stability_after": round(sim_after, 4)
                    })

# 4. Save and Export Results to Google Drive
output_file = '/content/vision_boundaries_multiframe.jsonl'
with open(output_file, 'w') as f:
    for boundary in segment_boundaries:
        f.write(json.dumps(boundary) + '\n')

shutil.copy(output_file, '/content/drive/MyDrive/vision_boundaries_multiframe.jsonl')
print(f"Advanced Multi-Frame Processing complete! Found {len(segment_boundaries)} verified state changes. Exported to Google Drive.")

Using compute device: cuda
Found 34563 screenshots. Commencing advanced multi-frame processing...
Advanced Multi-Frame Processing complete! Found 5894 verified state changes. Exported to Google Drive.
